# Замеры времени лучшей модели на всех датасетах

Используется модель, обученная в две стадии: сперва на датасете `synth_spell_correction_1m`, а затем на `spell_correction_30k`

In [ ]:
import time
import asyncio

import nest_asyncio
import pandas as pd
from tqdm.asyncio import tqdm_asyncio
from openai import AsyncOpenAI
from sage.utils import load_available_dataset_from_hf, DatasetsAvailable

# В Jupyter уже работает event loop, поэтому нужен nest_asyncio
nest_asyncio.apply()

In [ ]:
# ─── Настройки ───────────────────────────────────────────────────────────────
BASE_URL = "http://localhost:9900/v1"
API_KEY = "1234"
MODEL = "spell_correction"
TEMPERATURE = 0.1

# Максимум одновременных запросов
MAX_CONCURRENT = 512

PROMPT_TEMPLATE = (
    "Исходный текст:\n{text}\n\nОтредактируй исходный текст, исправив ошибки.\n"
)

client = AsyncOpenAI(base_url=BASE_URL, api_key=API_KEY)

In [3]:
async def correct_text(sem: asyncio.Semaphore, text: str) -> str:
    """Отправляет один запрос к модели с ограничением параллелизма."""
    async with sem:
        response = await client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": PROMPT_TEMPLATE.format(text=text)}],
            temperature=TEMPERATURE,
        )
    return response.choices[0].message.content


async def run_corrections(sources: list[str]) -> list[str]:
    """Запускает все запросы асинхронно с progress bar."""
    sem = asyncio.Semaphore(MAX_CONCURRENT)
    tasks = [correct_text(sem, s) for s in sources]
    return await tqdm_asyncio.gather(*tasks, desc="Inferencing")

In [4]:
def load_dataset(name: str):
    """
    Возвращает (sources, corrections) для любого датасета.
    RUSpellRU возвращает два списка напрямую,
    остальные — DataFrame с колонками source/correction.
    """
    if (
        name == DatasetsAvailable.RUSpellRU.name
        or name == DatasetsAvailable.MultidomainGold.name
    ):
        sources, corrections = load_available_dataset_from_hf(
            name, for_labeler=True, split="test"
        )
        return list(sources), list(corrections)
    else:
        df = load_available_dataset_from_hf(name, for_labeler=False)
        return df["source"].tolist(), df["correction"].tolist()

In [5]:
DATASET_NAMES = [d.name for d in DatasetsAvailable]
print("Датасеты:", DATASET_NAMES)

Датасеты: ['MultidomainGold', 'RUSpellRU', 'MedSpellchecker', 'GitHubTypoCorpusRu', 'MultidomainGold_orth', 'RUSpellRU_orth', 'MedSpellchecker_orth', 'GitHubTypoCorpusRu_orth']


In [6]:
DATASET_NAMES = [
    "MultidomainGold",
    "RUSpellRU",
    "MedSpellchecker",
    "GitHubTypoCorpusRu",
]

In [ ]:
OUTPUT_DIR = "../data/pred_data"

In [ ]:
for ds_name in DATASET_NAMES:
    print(f"\n{'=' * 60}")
    print(f"Датасет: {ds_name}")

    sources, corrections = load_dataset(ds_name)
    print(f"Примеров: {len(sources)}")

    # Запускаем асинхронный инференс с замером времени
    start_time = time.time()
    predictions = asyncio.run(run_corrections(sources))
    end_time = time.time()

    elapsed_time = end_time - start_time
    print(f"⏱️ Время выполнения инференса: {elapsed_time:.2f} секунд")
    print(f"📊 Среднее время на пример: {elapsed_time / len(sources):.3f} секунд")

    # Создаём DataFrame с результатами
    df_results = pd.DataFrame(
        {
            "source_text": sources,  # Исходный текст с ошибками
            "ground_truth": corrections,  # Эталонное исправление
            "model_prediction": predictions,  # Предсказание модели
        }
    )

    # Сохраняем полные результаты в CSV
    csv_path = f"{OUTPUT_DIR}/{ds_name}_predictions.csv"
    df_results.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"📁 Полные результаты сохранены в: {csv_path}")


Датасет: MultidomainGold
Примеров: 4106


Inferencing: 100%|██████████| 4106/4106 [01:15<00:00, 54.58it/s] 


⏱️ Время выполнения инференса: 75.24 секунд
📊 Среднее время на пример: 0.018 секунд
📁 Полные результаты сохранены в: data/pred_data/MultidomainGold_predictions.csv

Датасет: RUSpellRU
Примеров: 2008


Inferencing: 100%|██████████| 2008/2008 [00:45<00:00, 44.46it/s] 


⏱️ Время выполнения инференса: 45.17 секунд
📊 Среднее время на пример: 0.022 секунд
📁 Полные результаты сохранены в: data/pred_data/RUSpellRU_predictions.csv

Датасет: MedSpellchecker
Примеров: 1054


Inferencing: 100%|██████████| 1054/1054 [00:09<00:00, 105.87it/s]


⏱️ Время выполнения инференса: 9.96 секунд
📊 Среднее время на пример: 0.009 секунд
📁 Полные результаты сохранены в: data/pred_data/MedSpellchecker_predictions.csv

Датасет: GitHubTypoCorpusRu
Примеров: 868


Inferencing: 100%|██████████| 868/868 [00:09<00:00, 94.40it/s] 

⏱️ Время выполнения инференса: 9.21 секунд
📊 Среднее время на пример: 0.011 секунд
📁 Полные результаты сохранены в: data/pred_data/GitHubTypoCorpusRu_predictions.csv


В результате получили следующие результаты:
- Датасет: RUSpellRU
- Примеров: 2008
- ⏱️ Время выполнения инференса: 45.17 секунд
- 📊 Среднее время на пример: 0.022 секунд